# Agent 2 — Step B: Feature Engineering + Train/Test Split

**Input:** `master_prices.csv` (1853 rows from Step A)

**Output:** `train.csv`, `test.csv` with engineered features ready for XGBoost.

**What this does:**
1. Expand dataset with Grade A/B/C using APMC auction multipliers (×1.10 / ×1.00 / ×0.88)
2. Add cyclic month features (sin/cos) for seasonality
3. Encode categorical features (millet, state, district, grade)
4. Time-based split: train on 2021–2024, test on 2025–2026 (realistic forward-prediction)

## Cell 1 — Load master_prices.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np

DRIVE = "/content/drive/MyDrive/MilletSaarthi"
df = pd.read_csv(f"{DRIVE}/master_prices.csv")
print("Loaded rows:", len(df))
print(df.head())

## Cell 2 — Expand with Grade A/B/C (triples dataset)

Grade multipliers come from APMC auction pattern:
- **Grade A** = clean, <2% broken → +10% premium
- **Grade B** = standard, 2–5% broken → baseline (what Agmarknet modal price represents)
- **Grade C** = poor, >5% broken → −12% discount

In [ ]:
GRADE_MULT = {"A": 1.10, "B": 1.00, "C": 0.88}

expanded = []
for _, row in df.iterrows():
    for grade, mult in GRADE_MULT.items():
        new_row = row.copy()
        new_row["grade"] = grade
        new_row["modal_price"] = round(row["modal_price"] * mult, 2)
        expanded.append(new_row)

df = pd.DataFrame(expanded).reset_index(drop=True)
print("Expanded rows:", len(df))
print(df["grade"].value_counts())
print(df.head(6))

## Cell 3 — Feature engineering

In [ ]:
# Cyclic month features capture seasonality (Jan close to Dec)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Season tag (useful for explainability later)
def season(m):
    if m in (6, 7, 8, 9): return "kharif"
    if m in (10, 11, 12, 1, 2, 3): return "rabi"
    return "summer"
df["season"] = df["month"].apply(season)

# Label-encode categoricals (XGBoost handles integer categories well)
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ["millet", "state", "district", "grade", "season"]:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values")

# Save encoders for inference later
import pickle
with open(f"{DRIVE}/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
print("\n✅ Saved encoders.pkl")

## Cell 4 — Time-based train/test split

Train on **2021–2024**, test on **2025–2026** — simulates real forward prediction.

In [ ]:
FEATURES = [
    "millet_enc", "state_enc", "district_enc", "grade_enc", "season_enc",
    "year", "month", "month_sin", "month_cos"
]
TARGET = "modal_price"

train = df[df["year"] <= 2024].copy()
test  = df[df["year"] >= 2025].copy()

print(f"Train rows: {len(train)}  (years {train['year'].min()}–{train['year'].max()})")
print(f"Test  rows: {len(test)}  (years {test['year'].min()}–{test['year'].max()})")

train.to_csv(f"{DRIVE}/train.csv", index=False)
test.to_csv(f"{DRIVE}/test.csv", index=False)

# Save feature list for Step C
with open(f"{DRIVE}/features.txt", "w") as f:
    f.write(",".join(FEATURES))

print("\n✅ Saved train.csv, test.csv, features.txt")
print("\nTrain sample:")
print(train[FEATURES + [TARGET]].head())